# Tahap 1: Membangun Case Base (Mode Manual)
**Tujuan:** Konversi PDF yang sudah didownload manual → teks bersih

> ⚠️ Mode ini digunakan karena situs MA RI memblokir scraping otomatis (Error 403).  
> Pastikan sudah download ≥ 30 file PDF ke folder `data/raw/` sebelum menjalankan notebook ini.

In [ ]:
import os
import re
import logging
import glob
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError
import warnings
warnings.filterwarnings("ignore")
print("✅ Library berhasil diimport")

In [ ]:
# Konfigurasi Path
# Sesuaikan BASE_DIR dengan lokasi folder project kamu
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
LOG_DIR  = os.path.join(BASE_DIR, "logs")
os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

logging.basicConfig(
    filename=os.path.join(LOG_DIR, "cleaning.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

print(f"✅ Konfigurasi selesai")
print(f"   BASE_DIR : {BASE_DIR}")
print(f"   RAW_DIR  : {RAW_DIR}")
print(f"   LOG_DIR  : {LOG_DIR}")

## 📂 Cek Isi Folder data/raw/
Jalankan cell ini untuk melihat berapa PDF yang sudah ada.

In [ ]:
pdf_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.pdf")))
txt_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.txt")))

print(f"📂 Isi folder data/raw/:")
print(f"   PDF ditemukan : {len(pdf_files)} file")
print(f"   TXT ditemukan : {len(txt_files)} file")

if len(pdf_files) == 0 and len(txt_files) == 0:
    print("\n❌ FOLDER KOSONG!")
    print("   Download dulu ≥ 30 PDF dari:")
    print("   https://putusan3.mahkamahagung.go.id/direktori")
    print("   Cari: korupsi kerugian keuangan negara")
    print(f"   Simpan di: {RAW_DIR}")
else:
    print("\n📋 File yang ditemukan:")
    for f in (pdf_files + txt_files)[:10]:
        print(f"   📄 {os.path.basename(f)}")
    total = len(pdf_files) + len(txt_files)
    if total > 10:
        print(f"   ... dan {total - 10} file lainnya")

## 🔄 Rename PDF Otomatis (Opsional)
Jalankan cell ini **hanya jika** nama file PDF masih acak/panjang.  
Akan di-rename jadi `case_001.pdf`, `case_002.pdf`, dst.

In [ ]:
def rename_pdfs_otomatis():
    all_pdfs = sorted(glob.glob(os.path.join(RAW_DIR, "*.pdf")))
    if not all_pdfs:
        print("Tidak ada PDF untuk di-rename.")
        return
    renamed = 0
    for idx, old_path in enumerate(all_pdfs, start=1):
        new_name = f"case_{idx:03d}.pdf"
        new_path = os.path.join(RAW_DIR, new_name)
        if os.path.basename(old_path) != new_name:
            os.rename(old_path, new_path)
            print(f"  ✅ {os.path.basename(old_path)[:50]} → {new_name}")
            renamed += 1
        else:
            print(f"  ✔️  {new_name} sudah benar")
    print(f"\n  Total di-rename: {renamed} file")

# Uncomment baris di bawah untuk rename otomatis:
# rename_pdfs_otomatis()
print("Fungsi rename siap. Uncomment baris terakhir jika ingin dijalankan.")

## ⚙️ Fungsi Ekstraksi & Pembersihan Teks

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Ekstrak plain text dari file PDF menggunakan pdfminer."""
    try:
        text = extract_text(pdf_path)
        return text if text else ""
    except PDFSyntaxError:
        logger.error(f"PDF korup: {pdf_path}")
        return ""
    except Exception as e:
        logger.error(f"Error ekstrak {pdf_path}: {e}")
        return ""

def clean_text(raw_text: str) -> str:
    """
    Bersihkan teks hasil ekstraksi PDF:
    - Hapus header/footer/watermark MA RI
    - Normalisasi spasi dan karakter
    - Lowercase
    """
    # Hapus karakter non-ASCII
    text = re.sub(r"[^\x20-\x7E\n\t]", " ", raw_text)

    # Hapus watermark & header MA RI
    text = re.sub(r"mahkamah\s*agung\s*republik\s*indonesia", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"direktori\s*putusan", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"putusan\.mahkamahagung\.go\.id", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"disclaimer.*?kepaniteraan.*?\n", " ", text, flags=re.IGNORECASE)

    # Hapus nomor halaman
    text = re.sub(r"-\s*\d+\s*-", " ", text)
    text = re.sub(r"halaman\s*\d+\s*(dari\s*\d+)?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"hal\.\s*\d+", " ", text, flags=re.IGNORECASE)

    # Hapus baris kosong berlebihan
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Normalisasi spasi & lowercase
    text = re.sub(r"[ \t]+", " ", text)
    text = text.lower().strip()
    return text

def validate_text(text: str, min_words: int = 200) -> bool:
    """Cek teks minimal 200 kata."""
    return len(text.split()) >= min_words

print("✅ Fungsi ekstraksi & pembersihan siap")

## ▶️ Jalankan Pipeline: PDF → TXT
Cell ini akan mengkonversi semua PDF di folder `data/raw/` menjadi file teks bersih.

In [ ]:
pdf_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.pdf")))

if not pdf_files:
    print("❌ Tidak ada file PDF!")
    print(f"   Simpan PDF di: {RAW_DIR}")
else:
    print(f"{'='*60}")
    print(f"  KONVERSI {len(pdf_files)} FILE PDF → TXT")
    print(f"{'='*60}\n")

    berhasil, gagal, invalid = 0, 0, 0

    for idx, pdf_path in enumerate(pdf_files, start=1):
        filename = os.path.basename(pdf_path)
        case_id  = os.path.splitext(filename)[0]
        txt_path = os.path.join(RAW_DIR, f"{case_id}.txt")

        print(f"  [{idx:02d}/{len(pdf_files)}] {filename}", end=" → ")

        # Skip jika .txt sudah ada
        if os.path.exists(txt_path):
            print("sudah ada (skip)")
            berhasil += 1
            continue

        # Ekstrak teks
        raw_text = extract_text_from_pdf(pdf_path)
        if not raw_text.strip():
            print("❌ Gagal ekstrak teks")
            logger.warning(f"{case_id}: gagal ekstrak")
            gagal += 1
            continue

        # Bersihkan
        clean = clean_text(raw_text)

        # Validasi
        word_count = len(clean.split())
        if not validate_text(clean):
            print(f"⚠️  Terlalu pendek ({word_count} kata)")
            logger.warning(f"{case_id}: hanya {word_count} kata")
            invalid += 1
            continue

        # Simpan .txt
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(clean)

        logger.info(f"{case_id}: OK — {word_count} kata")
        print(f"✅ {word_count} kata")
        berhasil += 1

## 📊 Laporan Hasil

In [ ]:
txt_hasil = sorted(glob.glob(os.path.join(RAW_DIR, "*.txt")))

print(f"\n{'='*60}")
print(f"  RINGKASAN TAHAP 1")
print(f"{'='*60}")
print(f"  ✅ Berhasil dikonversi : {berhasil}")
print(f"  ❌ Gagal ekstrak       : {gagal}")
print(f"  ⚠️  Teks tidak valid   : {invalid}")
print(f"  📄 Total file .txt    : {len(txt_hasil)}")
print(f"  📁 Lokasi             : {RAW_DIR}")

if len(txt_hasil) >= 30:
    print(f"\n✅ {len(txt_hasil)} dokumen siap! Lanjut ke Tahap 2 (02_representation.ipynb)")
else:
    kurang = 30 - len(txt_hasil)
    print(f"\n⚠️  Baru {len(txt_hasil)} dokumen. Butuh {kurang} PDF lagi.")
    print(f"   Download lebih banyak dari situs MA RI!")
print(f"{'='*60}")